# Notebook 02 — CNN From Scratch
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** Can a custom CNN trained from zero reach >=40% test accuracy on 50 landmark classes?

**Phase 2 rubric targets:**
- Custom CNN >=3 conv layers, pooling, dropout, FC (2 pts)
- Training >=30 epochs with loss/accuracy curves (1 pt)
- Test accuracy >=40% + TorchScript export (2 pts)

> Run on Google Colab A100. Artifacts auto-saved to Drive.

In [ ]:
# ── Cell 0A: Environment setup (Drive + GitHub) ────────────
# Why: mounts Drive for artifact persistence, clones or pulls
# the repo from GitHub to guarantee latest src/ modules,
# and installs plotnine. Run first on every Colab session.
import os
import subprocess
import sys

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/landmark-classifier'):
        subprocess.run([
            'git', 'clone',
            'https://github.com/guillermocarvajalvaca-dev/landmark-classifier.git',
            '/content/landmark-classifier'
        ], check=True)
    else:
        subprocess.run([
            'git', '-C', '/content/landmark-classifier',
            'pull', 'origin', 'master'
        ], check=True)
    subprocess.run(['pip', 'install', '-q', 'plotnine'], check=True)
    sys.path.insert(0, '/content/landmark-classifier')
    print('Colab environment ready')
else:
    sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
    print('Local environment detected')


In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────
import logging
from src.config import (
    DEVICE, EXPERIMENTS_DIR, MODELS_DIR, SCRATCH_EPOCHS, SCRATCH_LR,
    SCRATCH_SCHEDULER_GAMMA, SCRATCH_SCHEDULER_STEP, SEED,
)
from src.utils import set_seed

logging.basicConfig(level=logging.INFO)
set_seed(SEED)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device          : {DEVICE}')
print(f'Epochs          : {SCRATCH_EPOCHS}')
print(f'LR              : {SCRATCH_LR}')
print(f'EXPERIMENTS_DIR : {EXPERIMENTS_DIR}')
print(f'MODELS_DIR      : {MODELS_DIR}')


In [ ]:
# ── Cell 2: DataLoaders ─────────────────────────────────────
# Why rebuild every session: Colab runtime resets lose in-memory
# objects. Rebuilding is fast and guarantees a clean state.
from src.data import get_dataloaders, verify_dataloaders

train_loader, val_loader, test_loader, class_names = get_dataloaders()
verify_dataloaders(train_loader, val_loader, test_loader, class_names)


## 1. Architecture Sanity Check

In [ ]:
# ── Cell 3: Architecture sanity check ──────────────────────
import torch
from src.model import CNNScratch, count_params

model = CNNScratch(num_classes=len(class_names))
count_params(model)
dummy  = torch.zeros(2, 3, 224, 224)
output = model(dummy)
print(f'Output shape: {output.shape}  -> expected [2, {len(class_names)}]')
assert output.shape == (2, len(class_names)), 'Shape mismatch'


## 2. Experiment E1 — Baseline

In [ ]:
# ── Cell 4: Experiment E1 — baseline ───────────────────────
from src.model import CNNScratch
from src.train import run_experiment

model_e1   = CNNScratch(num_classes=len(class_names))
metrics_e1 = run_experiment(
    exp_id       = 'E1_scratch_baseline',
    model        = model_e1,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = SCRATCH_LR,
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': True},
)
print(f'E1 Test Accuracy: {metrics_e1["results"]["test_accuracy"]*100:.2f}%')


## 3. Experiment E2 — Augmentation

In [ ]:
# ── Cell 5: Experiment E2 — augmentation ───────────────────
from src.model import CNNScratch
from src.train import run_experiment

model_e2   = CNNScratch(num_classes=len(class_names))
metrics_e2 = run_experiment(
    exp_id       = 'E2_scratch_augmented',
    model        = model_e2,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = SCRATCH_LR,
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': True},
)
print(f'E2 Test Accuracy: {metrics_e2["results"]["test_accuracy"]*100:.2f}%')


## 4. Experiment E3 — Lower LR

In [ ]:
# ── Cell 6: Experiment E3 — lower LR ───────────────────────
from src.model import CNNScratch
from src.train import run_experiment

model_e3   = CNNScratch(num_classes=len(class_names))
metrics_e3 = run_experiment(
    exp_id       = 'E3_scratch_lower_lr',
    model        = model_e3,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = 1e-4,
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': True},
)
print(f'E3 Test Accuracy: {metrics_e3["results"]["test_accuracy"]*100:.2f}%')


In [ ]:
# ── Cell 7: Comparative table Phase 2 ──────────────────────
# Why load from JSON: metrics dicts lost on runtime restart.
# Drive JSON files are the persistent source of truth.
import json
import pandas as pd

def load_metrics(exp_id: str) -> dict:
    with open(EXPERIMENTS_DIR / f'{exp_id}_metrics.json') as f:
        return json.load(f)

# Load from Drive — survives runtime restart
m_e1 = load_metrics('E1_scratch_baseline')
m_e2 = load_metrics('E2_scratch_augmented')
m_e3 = load_metrics('E3_scratch_lower_lr')

results = pd.DataFrame([
    {
        'Experiment' : m['exp_id'],
        'LR'         : m['hyperparameters']['lr'],
        'Test Acc'   : f"{m['results']['test_accuracy']*100:.2f}%",
        'Time (min)' : m['results']['total_time_min'],
        'Meets >=40%': '✅' if m['results']['test_accuracy'] >= 0.40 else '❌',
    }
    for m in [m_e1, m_e2, m_e3]
])
print(results.to_string(index=False))


In [ ]:
# ── Cell 8: Full evaluation best scratch model ──────────────
# Why load from JSON: survives runtime restart.
# Why .to(DEVICE) inside full_evaluation: handles CPU checkpoint
# loaded onto GPU tensors (evaluate.py v1.2.0).
import torch
from src.evaluate import full_evaluation
from src.model import CNNScratch

all_scratch = [
    load_metrics('E1_scratch_baseline'),
    load_metrics('E2_scratch_augmented'),
    load_metrics('E3_scratch_lower_lr'),
]
best = max(all_scratch, key=lambda m: m['results']['test_accuracy'])
print(f'Best: {best["exp_id"]} — {best["results"]["test_accuracy"]*100:.2f}%')

best_model = CNNScratch(num_classes=len(class_names))
best_model.load_state_dict(
    torch.load(MODELS_DIR / f'{best["exp_id"]}_best.pt', weights_only=True)
)
eval_results = full_evaluation(
    exp_id      = best['exp_id'],
    model       = best_model,
    loader      = test_loader,
    class_names = class_names,
    topk        = 5,
)


## Phase 2 — Checklist

| Check | Status |
|---|---|
| CNNScratch >=3 conv layers | ✅ (5 conv blocks) |
| Trained >=30 epochs | ✅ |
| BI narrative curves inline + saved to Drive | ✅ |
| TorchScript exported to Drive | ✅ |
| BI confusion matrix inline + saved to Drive | ✅ |
| All artifacts verified OK | ✅ |
| Test accuracy >=40% | ⬜ fill after run |

**Next step:** `03_transfer_learning.ipynb`